In [ ]:
import os
import cv2
import numpy as np

def process_single_image(image_path: str, black_threshold: int = 20, min_area: int = 100) -> np.ndarray:
    img = cv2.imread(image_path, cv2.IMREAD_GRAYSCALE)
    if img is None:
        raise ValueError(f"Не удалось загрузить изображение: {image_path}")
    
    h, w = img.shape
    center_x, center_y = w / 2.0, h / 2.0
    mid_col = int(w // 2)

    #черные пиксели
    black_mask = (img <= black_threshold).astype(np.uint8) * 255

    #кластеры
    num_labels, labels, stats, centroids = cv2.connectedComponentsWithStats(
        black_mask, connectivity=8
    )
    
    left_best_label = None
    left_max_area = 0
    
    right_best_label = None
    right_max_area = 0

    #лучший кластер отдельно для левой и правой части
    for label in range(1, num_labels):
        area = stats[label, cv2.CC_STAT_AREA]
        
        #мелкий шум
        if area < min_area:
            continue
            
        cx, cy = centroids[label]
        
        #наибольший по площади кластер для каждой половины
        if cx < mid_col:
            if area > left_max_area:
                left_max_area = area
                left_best_label = label
        else:
            if area > right_max_area:
                right_max_area = area
                right_best_label = label

    #формируем результат
    result_img = img.copy()
    valid_labels = {left_best_label, right_best_label} - {None}

    #закрашиваем белым
    keep_mask = np.isin(labels, list(valid_labels))
    remove_mask = (black_mask > 0) & (~keep_mask)
    
    result_img[remove_mask] = 255

    return result_img


def process_folder(input_folder: str, output_folder: str, black_threshold: int = 20):
    """
    Пакетная обработка всех изображений в папке input_folder
    и сохранение результатов в output_folder.
    """
    if not os.path.exists(output_folder):
        os.makedirs(output_folder)

    valid_extensions = ('.png', '.jpg', '.jpeg', '.bmp', '.tiff')
    files = [f for f in os.listdir(input_folder) if f.lower().endswith(valid_extensions)]

    print(f"Найдено изображений для обработки: {len(files)}")

    for filename in files:
        in_path = os.path.join(input_folder, filename)
        out_path = os.path.join(output_folder, filename)

        try:
            processed_img = process_single_image(in_path, black_threshold=black_threshold)
            cv2.imwrite(out_path, processed_img)
        except Exception as e:
            print(f"Ошибка при обработке файла {filename}: {e}")

    print("Обработка завершена!")


#пример использования:
#process_folder(input_folder="path/to/raw_dataset", output_folder="path/to/cleaned_dataset")